Step 1: Read and prepare the data

1.1 Read the gene expression matrix

First, read the gene expression matrix file and transpose it so that the gene expression values of each sample become the input to the model.

In [5]:
SESSION_ID = 2026

In [ ]:
import pandas as pd

# Read gene expression matrix
gene_expression_df=pd.read_csv('/home/chenliqun/Tcga/input/newtop50_matrix.txt',sep='\t',index_col=0)

# Transpose matrix: rows become samples, columns become genes
X=gene_expression_df.transpose()  # shape=(2959,219)
print(X.shape)

(2959, 219)


1.2 Read tumor type labels

Next, read the tumor type label file. This file contains the tumor type for each sample (the `sourced_tumor_name` column), and it should be aligned with the samples in the gene expression matrix.


In [ ]:
labels_df = pd.read_csv('/home/chenliqun/Tcga/input/Final_TCGA_sourcetumornames_selected7tumors_TcgaTargetGTEX_phenotype_filtered_samples.txt', sep='\t')
labels_df.set_index('sample', inplace=True) 
y = labels_df['sourced_tumor_name']  # shape = (3280,)
print(y.shape)

(2959,)


1.3 Data validation

Verify that the sample order in the gene expression matrix matches the sample order in the label file. If the sample orders are different, you may need to align them using the sample IDs.


In [ ]:
y=y.reindex(X.index)
missing_labels=y.isnull().sum()
if missing_labels>0:
    print(f"Warning: {missing_labels} samples have no corresponding labels in the label file")
    # Remove samples without labels
    X=X[y.notnull()]
    y=y.dropna()

# Validation
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Number of samples: {len(X)}")
print("First 5 sample labels:")
print(y.head())

# Assertion check
assert X.index.equals(y.index),"Sample order mismatch!"
print("✅ Sample order aligned!")

X形状: (2959, 219)
y形状: (2959,)
样本数: 2959
前5个样本标签:
TCGA-B5-A5OE-01               Endometrial carcinoma
TCGA-C8-A1HL-01           ER positive breast cancer
TCGA-EW-A2FS-01           ER positive breast cancer
TCGA-IR-A3L7-01    Cervical squamous cell carcinoma
TCGA-B6-A402-01           ER positive breast cancer
Name: sourced_tumor_name, dtype: object
✅ 样本顺序已对齐！


In [ ]:
print(y.head(2))
print(X.iloc[:2, :2])
print(y.index)
print(X.index)

assert X.index.equals(y.index), "Sample order mismatch! Please check the data."


TCGA-B5-A5OE-01        Endometrial carcinoma
TCGA-C8-A1HL-01    ER positive breast cancer
Name: sourced_tumor_name, dtype: object
sample            PITX3     RAX
TCGA-B5-A5OE-01 -2.1140 -6.5064
TCGA-C8-A1HL-01 -4.2934 -9.9658
Index(['TCGA-B5-A5OE-01', 'TCGA-C8-A1HL-01', 'TCGA-EW-A2FS-01',
       'TCGA-IR-A3L7-01', 'TCGA-B6-A402-01', 'TCGA-D5-6929-01',
       'TCGA-AX-A3G6-01', 'TCGA-A2-A3XX-01', 'TCGA-BP-4765-01',
       'TCGA-CZ-5459-01',
       ...
       'TCGA-5T-A9QA-01', 'TCGA-AN-A0XT-01', 'TCGA-DM-A28A-01',
       'TCGA-G2-AA3C-01', 'TCGA-A2-A0YK-01', 'TCGA-AA-3660-01',
       'TCGA-A8-A09K-01', 'TCGA-B5-A3S1-01', 'TCGA-A2-A1FV-01',
       'TCGA-FI-A2EY-01'],
      dtype='object', length=2959)
Index(['TCGA-B5-A5OE-01', 'TCGA-C8-A1HL-01', 'TCGA-EW-A2FS-01',
       'TCGA-IR-A3L7-01', 'TCGA-B6-A402-01', 'TCGA-D5-6929-01',
       'TCGA-AX-A3G6-01', 'TCGA-A2-A3XX-01', 'TCGA-BP-4765-01',
       'TCGA-CZ-5459-01',
       ...
       'TCGA-5T-A9QA-01', 'TCGA-AN-A0XT-01', 'TCGA-DM-A28A-01'

2.1 Split training and test sets

We use `train_test_split` to divide the data into training and test sets.
80% of the data is used for training and 20% for testing, and `stratify` is used to ensure that the class proportions remain consistent.


In [ ]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SESSION_ID)
out_dir = "/home/chenliqun/Tcga/splits_data"
os.makedirs(out_dir, exist_ok=True)

pd.Series(X_train.index, name="sample").to_csv(f"{out_dir}/train_idx.csv", index=False)
pd.Series(X_test.index,  name="sample").to_csv(f"{out_dir}/test_idx.csv",  index=False)

X_train.to_csv(f"{out_dir}/X_train.tsv", sep="\t", index=True)
X_test.to_csv(f"{out_dir}/X_test.tsv", sep="\t", index=True)

y_train.to_frame("target").to_csv(f"{out_dir}/y_train.tsv", sep="\t", index=True)
y_test.to_frame("target").to_csv(f"{out_dir}/y_test.tsv", sep="\t", index=True)

print("✅ Saved split to:", out_dir)
print("Train:", X_train.shape, "Test:", X_test.shape)


✅ Saved split to: /home/chenliqun/Tcga/splits_data
Train: (2367, 219) Test: (592, 219)


In [12]:
from sklearn.model_selection import KFold
import os
import pandas as pd

cv_dir = "/home/chenliqun/Tcga/splits_data/cv5"
os.makedirs(cv_dir, exist_ok=True)

kf = KFold(n_splits=5, shuffle=True, random_state=SESSION_ID)

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), start=1):
    train_samples = X_train.index[train_idx]
    val_samples   = X_train.index[val_idx]

    pd.Series(train_samples, name="sample").to_csv(
        f"{cv_dir}/fold{fold}_train_idx.csv",
        index=False
    )

    pd.Series(val_samples, name="sample").to_csv(
        f"{cv_dir}/fold{fold}_valid_idx.csv",
        index=False
    )

print("✅ 5-fold CV indices saved (no subfolders).")


✅ 5-fold CV indices saved (no subfolders).
